In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from lesson_1.ingest import load_faq_data
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
from elasticsearch import Elasticsearch

In [ ]:
es = Elasticsearch("http://localhost:9200")

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
documents = load_faq_data()
documents[:2]

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In [32]:
texts = [doc["question"].removeprefix('Course: ') + " " + doc["answer"] for doc in documents]
texts[:3]

["When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 "What are the prerequisites for this course? To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up front. The cour

In [ ]:
#chunk the dataset (texts) by batch. There are 27 batches given texts size is 13++ and each batch is 50. Encode by batch. 
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
len(vectors), len(vectors[0])

  0%|          | 0/27 [00:00<?, ?it/s]

(1349, 384)

In [36]:
X = np.array(vectors)
X.shape #1346 vectors (rows) with 384 dimensions (columns)

# Each vector is the semantic "fingerprint" of its corresponding text chunk. So:
    # vectors[0] captures the meaning of the course start date chunk
    # vectors[1] captures the meaning of the prerequisites chunk

(1349, 384)

In [63]:
dimension = len(X[0])

def create_vector_index(documents, X):

    # Delete first in case the index existed:
    es.indices.delete(index="embedded_faq_zoomcamp", ignore_unavailable=True)

    # Create index:
    index_settings = {
        "mappings": {
            "properties": {
                "course":   {"type": "keyword"}
                ,"Q": {"type": "text"}
                ,"A":   {"type": "text"}
                ,"vector": {
                    "type":       "dense_vector"
                    ,"dims":       len(X[0])
                    ,"index":      True
                    ,"similarity": "cosine"
                }
            }
        }
    }

    es.indices.create(index="embedded_faq_zoomcamp", body=index_settings)

    # Index documents with vectors:
    for i, doc in enumerate(documents):
        es.index(
            index="embedded_faq_zoomcamp",
            id=doc["id"],
            body={
                "course": doc["course"]
                ,"Q": doc["question"].removeprefix('Course: ')
                ,"A": doc["answer"]
                ,"vector": X[i].tolist()
            }
        )
    
    # Force refresh so all docs are committed before counting
    es.indices.refresh(index="embedded_faq_zoomcamp")

    doc_count = es.count(index="embedded_faq_zoomcamp")['count']
    
    print(f'Vector index created and indexed {doc_count} docs')

In [64]:
create_vector_index(documents, X)

Vector index created and indexed 1349 docs


In [76]:
filter_dict = {"course": "llm-zoomcamp"}

response = es.search(
    index="embedded_faq_zoomcamp"
    ,body={
        "query": {
            "bool": {
                "must": {"match_all": {}}
                ,"filter": {
                    "term": filter_dict
                }
            }
        }
        ,"size": 2
    }
)

for hit in response["hits"]["hits"]:
    print(hit["_id"],hit["_score"])
    print(hit["_source"])
    print("---")

74eb249bbf 1.0
{'course': 'llm-zoomcamp', 'Q': 'I just discovered the course. Can I still join?', 'A': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'vector': [-0.0599796324968338, -0.057748597115278244, -0.030979547649621964, 0.019150346517562866, 0.024852653965353966, -0.019456537440419197, -0.08058660477399826, -0.06111559271812439, -0.06444694846868515, -0.04227716848254204, -0.042185936123132706, -0.012241702526807785, 0.04257587343454361, 0.030094128102064133, 0.0157663244754076, -0.03331267088651657, -0.08642635494470596, -0.02962839975953102, 0.05685659125447273, 0.006621004082262516, -0.05752572417259216, 0.03197608143091202, -0.017901478335261345, 0.009050622582435608, -0.014255544170737267, -0.02193446457386017, 0.03953805938363075, 0.08630134165287018, 0.006488591432571411, -0.03575819358229637, -0.03371449559926987, 0.042401596903800964, 0.028510579839348793, 0.009236178360879421, 0.02550216205418

In [70]:
response = es.search(
    index="embedded_faq_zoomcamp"
    ,body={
        "query": {"match_all": {}}
        ,"size": 2
    }
)

for hit in response["hits"]["hits"]:
    print(hit["_source"])
    print("---")

{'course': 'data-engineering-zoomcamp', 'Q': 'When does the course start?', 'A': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.", 'vector': [-0.02720794640481472, -0.13237154483795166, 0.016935553401708603, 0.003606025828048587, -0.02335505746304989, -0.060491230338811874, -0.08714234828948975, -0.01158521044999361, -0.0969802737236023, 0.03208034113049507, 0.032273758202791214, -0.0023511853069067, -0.030383042991161346, -0.026810025796294212, 0.03182227909564972, -0.016380613669753075, 0.014796162024140358, -0.17487291991710663, 0.06387914717197418, -0.0